[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Incertidumbre, intervalos de confianza y *bootstrap*.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete
**Lunes 26 de octubre de 2026 · 6:30 a 8:30 pm (CDMX)** · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En esta sesión se cuantificará la incertidumbre de una estimación: se simulará la distribución muestral con *NumPy*, se calcularán errores estándar e intervalos de confianza para medias y proporciones con *SciPy* y *statsmodels*, y se construirán intervalos *bootstrap* con `scipy.stats.bootstrap`, usando `penguins`. Al final se presenta la consigna del **Proyecto integrador**.

## 🎯 Objetivo de la sesión.

**Acompañar cada estimación con su incertidumbre, con fórmulas clásicas y con remuestreo, e interpretarla correctamente.**

Al terminar podrás:
1. Explicar la distribución muestral, la ley de los grandes números y el teorema central del límite, y cómo el tamaño de muestra y el diseño (estratificado o aleatorio simple) cambian el error estándar.
2. Calcular intervalos de confianza para una media (fórmula *t*) y para una proporción (Wald, Wilson, Clopper-Pearson).
3. Construir intervalos *bootstrap* para la media, la mediana y la diferencia entre dos grupos.
4. Interpretar un intervalo de confianza sin las confusiones más comunes y elegir el método según los datos.

## 🗺️ Agenda (120 min).

| Min | Bloque |
|---|---|
| 10 | Retroalimentación de la Evaluación semanal 1 |
| 30 | 1. Distribución muestral: ley de los grandes números, teorema central del límite, tamaño de muestra y diseño |
| 25 | 2. Error estándar e intervalos de confianza |
| 25 | 3. *Bootstrap* |
| 15 | 4. Interpretación correcta de un IC y consigna del Proyecto integrador |
| 20 | 5. Ejercicio de la sesión y cierre |

🟢 = ejercicio básico · 🔴 = reto opcional · 📖 = lectura opcional (si el tiempo no alcanza, se revisa en casa)

## 📚 Referencias.

* La generación de números aleatorios con `np.random.default_rng` se explica en [py311 · 04 Arreglos con contenido aleatorio](https://github.com/PythonistaMX/py311/blob/main/04_arreglos_con_contenido_aleatorio.ipynb) y los resúmenes con *NumPy* en [py311 · 07 Gestión y análisis de datos de *NumPy*](https://github.com/PythonistaMX/py311/blob/main/07_gestion_y_analisis_de_datos_de_numpy.ipynb).
* py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html), [*SciPy* · `bootstrap`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html) y [*statsmodels* · `proportion_confint`](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportion_confint.html).

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3; si algo se ve distinto, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, incluidos `stats` de *SciPy* y las funciones de *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.weightstats import DescrStatsW

**¿Por qué este paso?** Se parte del archivo limpio de la sesión 2 para no repetir la limpieza; si no existe (por ejemplo, en un *Codespace* nuevo), se regenera con los mismos pasos mínimos. `titanic` se usará para proporciones y para un ejemplo de datos muy asimétricos.

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará desde `penguins.csv` si no existe) y `titanic.csv`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
pinguinos.shape, titanic.shape

---
## 0. Retroalimentación de la Evaluación semanal 1 (10 min).

El instructor comentará en vivo los resultados del grupo. Estos son los puntos que más conviene revisar en cualquier entrega de EDA descriptivo:

| Aspecto | Error frecuente | Lo que se espera |
|---|---|---|
| Calidad | Revisar faltantes pero no duplicados ni `dtypes` | Lista de verificación completa de la sesión 2 |
| Descripción | Reportar solo media y desviación estándar de `tip_pct`, que es asimétrica y tiene atípicos | Mediana e IQR, o ambas con la justificación |
| Agregación | Comparar medias por grupo sin el número de cuentas de cada grupo | Tabla con *n* por celda (en `tips`, las comidas son casi todas en jueves) |
| Gráficas | Barras de medias sin dispersión, ejes sin unidades | Diagramas de caja o puntos con dispersión; títulos con el hallazgo |
| Conclusiones | "Los fumadores dejan más propina porque…" | Lenguaje descriptivo: "en estos datos, la mediana fue…" |

Hoy se agrega la pieza que faltaba en la evaluación 1: **¿qué tan seguros estamos de esos números?**

---
## 1. Población, muestra y distribución muestral (30 min).

En la sesión 1 se distinguieron **parámetros** (valores de la población, como $\mu$ y $\sigma$) y **estadísticos** (valores calculados con la muestra, como $\bar{x}$ y $s$). Un estadístico es una **variable aleatoria**: si se repitiera el muestreo, saldría otro valor. Toda la inferencia de las sesiones 5 y 6 descansa en una pregunta: **¿cuánto cambiaría el estadístico de una muestra a otra?**

* **Población:** todos los individuos de interés. Su media $\mu = \frac{1}{N}\sum_{i=1}^{N} x_i$ es desconocida.
* **Muestra:** los $n$ individuos medidos. Su media $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$ es un **estimador** de $\mu$.
* **Distribución muestral:** la distribución de $\bar{x}$ sobre todas las muestras posibles de tamaño $n$. Su desviación estándar es el **error estándar** (EE):

$$EE(\bar{x}) = \frac{\sigma}{\sqrt{n}}$$

No hay que confundir la **desviación estándar** $\sigma$ (cuánto varían los **individuos**) con el **error estándar** (cuánto varía la **estimación**).

Para **ver** la distribución muestral se necesita conocer la población. Por eso, solo en esta sección, se tratarán los 342 pingüinos con masa registrada como si fueran la población completa y se sacarán muchas muestras de ella. Se muestrea **con reemplazo**, como si la población fuera muy grande; así las fórmulas son exactas.

* La siguiente celda definirá la población de juguete y su media y desviación estándar.

In [ ]:
poblacion = pinguinos["body_mass_g"].dropna().to_numpy()
mu, sigma = poblacion.mean(), poblacion.std()
round(float(mu), 1), round(float(sigma), 1)

**Interpretación:** en esta población de juguete, $\mu$ ≈ 4,201.8 g y $\sigma$ ≈ 800.8 g. Se usa `std()` de *NumPy* (con `ddof=0`) porque aquí se trata de la población completa, no de una muestra.

### 1.1 Ley de los grandes números.

La **ley de los grandes números** dice que, al aumentar el tamaño de la muestra, la media muestral se acerca al valor de la población:

$$\bar{x}_n \xrightarrow[n \to \infty]{} \mu$$

Es la justificación de que "más datos dan mejores estimaciones". Para verla, se sacará una sola secuencia larga de pingüinos y se calculará la **media acumulada** después de cada nuevo dato. Se usa un generador local con semilla 401 para que los resultados no dependan de las celdas anteriores.

* La siguiente celda sacará 5,000 pingüinos al azar, calculará la media acumulada y la dibujará frente a μ.

In [ ]:
generador_lgn = np.random.default_rng(401)
secuencia = generador_lgn.choice(poblacion, size=5000)
media_acumulada = np.cumsum(secuencia) / np.arange(1, len(secuencia) + 1)
ax = sns.lineplot(x=np.arange(1, len(secuencia) + 1), y=media_acumulada)
ax.axhline(mu, color="black", linestyle="--")
ax.set(xscale="log", xlabel="número de pingüinos observados (escala logarítmica)", ylabel="media acumulada (g)",
       title="Ley de los grandes números: la media acumulada se acerca a μ")
plt.show()
{n: round(float(media_acumulada[n - 1]), 1) for n in [1, 10, 100, 1000, 5000]}

**Interpretación:** con un pingüino la "media" es 3,475 g, a más de 700 g de $\mu$; con 100 está a unos 80 g, y con 5,000 a menos de 1 g. Las oscilaciones iniciales son grandes y se amortiguan rápido. **Error común que se evita:** la ley de los grandes números **no** dice que la muestra "compense" una racha (la falacia del jugador); dice que el peso de cada racha se diluye al crecer $n$. Tampoco dice **qué tan rápido** se acerca: eso lo responde el error estándar (sección 1.3).

### 1.2 Teorema central del límite.

El **teorema central del límite** (TCL) describe la **forma** de la distribución muestral: si las observaciones son independientes y tienen varianza finita, para $n$ suficientemente grande

$$\bar{x} \;\approx\; \mathcal{N}\!\left(\mu,\; \frac{\sigma^2}{n}\right)$$

sin importar la forma de la población. Es la razón por la que las fórmulas de intervalos y pruebas de las sesiones 5 y 6 usan la distribución normal o la *t*. Primero se verá con la masa de los pingüinos, una población **bimodal**.

* La siguiente celda sacará 2,000 muestras de tamaño 25 y guardará la media de cada una.

In [ ]:
medias_25 = rng.choice(poblacion, size=(2000, 25)).mean(axis=1)
round(float(medias_25.mean()), 1), round(float(medias_25.std()), 1)

* La siguiente celda comparará la distribución de la población con la distribución muestral de la media.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), sharex=True)
sns.histplot(poblacion, bins=30, ax=axes[0]).set(title="Población: masa de 342 pingüinos (bimodal)", xlabel="masa (g)")
sns.histplot(medias_25, bins=40, ax=axes[1]).set(title="Distribución muestral de la media (n = 25)", xlabel="media muestral (g)")
for ax in axes:
    ax.axvline(mu, color="black", linestyle="--")
plt.tight_layout()
plt.show()

**Interpretación:** aunque la población es bimodal, las medias de muestras de 25 se distribuyen de forma casi normal alrededor de $\mu$: es el TCL en acción. Su centro coincide con $\mu$ (el estimador es **insesgado**) y su dispersión, unos 160 g, es el error estándar: $800.8 / \sqrt{25} \approx 160$ g, cinco veces menor que la dispersión de los individuos.

**¿Qué tan grande debe ser *n*?** Depende de la población. La regla "con $n \geq 30$ basta" funciona con poblaciones parecidas a la normal, pero falla con poblaciones muy asimétricas. Se probará con la **tarifa del Titanic** (asimetría ≈ 4.8: muchas tarifas bajas y unas pocas altísimas), midiendo la asimetría de la distribución de las medias. Una distribución normal tiene asimetría 0.

* La siguiente celda dibujará la distribución de la media de la tarifa para muestras de 1, 5, 30 y 100 pasajeros, y calculará su asimetría y su error estándar simulado y teórico.

In [ ]:
generador_tcl = np.random.default_rng(401)
tarifas = titanic["fare"].to_numpy()
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
filas_tcl = {}
for ax, n_tcl in zip(axes, [1, 5, 30, 100]):
    medias_tarifa = generador_tcl.choice(tarifas, size=(5000, n_tcl)).mean(axis=1)
    sns.histplot(medias_tarifa, bins=50, ax=ax).set(title=f"n = {n_tcl}", xlabel="media de la tarifa", ylabel="")
    filas_tcl[n_tcl] = {"asimetría": stats.skew(medias_tarifa), "EE simulado": medias_tarifa.std(),
                        "σ / √n": tarifas.std() / np.sqrt(n_tcl)}
plt.tight_layout()
plt.show()
pd.DataFrame(filas_tcl).T.round(2)

**Interpretación:** con $n = 1$ la "distribución de la media" es la población misma (asimetría ≈ 4.4). La asimetría baja a ≈ 2.3 con $n = 5$, a ≈ 0.8 con $n = 30$ y a ≈ 0.6 con $n = 100$: se acerca a la normal, pero **lentamente**. En cambio, el error estándar simulado coincide con $\sigma / \sqrt{n}$ en todos los casos, porque esa fórmula no depende de la forma de la población.

**Decisión que permite tomar:** con datos tan asimétricos, $n = 30$ no basta para confiar en la aproximación normal; esto explica la mala cobertura que se verá en la sección 2.2. Conviene revisar la forma de los datos (sesión 3) antes de usar una fórmula basada en el TCL. **Error común que se evita:** creer que el TCL vuelve normales **los datos**; solo vuelve aproximadamente normal **la media**.

### 1.3 Efecto del tamaño de muestra: el error estándar baja con √*n*.

Como $EE = \sigma / \sqrt{n}$, cuadruplicar la muestra reduce el error estándar **a la mitad**, no a la cuarta parte: la precisión cuesta cada vez más datos. Para pasar de un EE de 80 g a uno de 40 g hacen falta 300 pingüinos más; para bajar a 20 g, 1,200 más.

* La siguiente celda comparará el error estándar simulado con la fórmula σ / √n para tres tamaños de muestra.

In [ ]:
pd.DataFrame({
    n: {"EE simulado": rng.choice(poblacion, size=(2000, n)).mean(axis=1).std(),
        "σ / √n": sigma / np.sqrt(n)}
    for n in [25, 100, 400]
}).T.round(1)

**Interpretación:** la simulación confirma la fórmula: ≈ 160, 80 y 40 g para 25, 100 y 400 pingüinos. Cada vez que *n* se multiplica por 4, el EE se divide entre 2. **Decisión que permite tomar:** dimensionar una muestra antes de recolectar datos. Si se quiere un EE de 50 g, hace falta $n \approx (800.8 / 50)^2 \approx 257$.

### 1.4 📖 Lectura opcional en clase: efecto del diseño (estratificado contra aleatorio simple).

En la sesión 1 se vio que estratificar por especie daba estimaciones más estables. Ahora se cuantificará. Si la población se divide en $H$ estratos con pesos $W_h = N_h / N$ y desviaciones estándar $\sigma_h$, la media **estratificada** es $\bar{x}_{st} = \sum_h W_h \bar{x}_h$ y su error estándar es

$$EE(\bar{x}_{st}) = \sqrt{\sum_{h=1}^{H} \frac{W_h^2\, \sigma_h^2}{n_h}}$$

Con asignación **proporcional** ($n_h = n W_h$), esto se simplifica a $\sqrt{\sum_h W_h \sigma_h^2 / n}$. Es decir, solo cuenta la variabilidad **dentro** de los estratos. La variabilidad **entre** estratos (que los Gentoo pesen 1,300 g más que los Adelie) desaparece del error. Por eso estratificar rinde mucho cuando los estratos difieren entre sí.

* La siguiente celda calculará los pesos y las desviaciones estándar de cada especie en la población.

In [ ]:
poblacion_especies = pinguinos.dropna(subset=["body_mass_g"])
pesos_estrato = poblacion_especies["species"].value_counts(normalize=True).sort_index()
sigma_estrato = poblacion_especies.groupby("species")["body_mass_g"].std(ddof=0)
masas_estrato = {especie: datos["body_mass_g"].to_numpy() for especie, datos in poblacion_especies.groupby("species")}
pd.DataFrame({"peso W_h": pesos_estrato, "σ_h (g)": sigma_estrato}).round(3)

**Interpretación:** las desviaciones estándar **dentro** de cada especie (380 a 500 g) son mucho menores que la de la población completa (800.8 g), porque buena parte de la variación total se debe a las diferencias **entre** especies.

* La siguiente celda simulará 5,000 muestras de cada diseño para *n* = 30, 60 y 120 y comparará los errores estándar simulados con sus fórmulas.

In [ ]:
generador_diseno = np.random.default_rng(401)
filas_diseno = {}
for n_total in [30, 60, 120]:
    n_estrato = {especie: int(round(n_total * peso)) for especie, peso in pesos_estrato.items()}
    medias_simple = generador_diseno.choice(poblacion, size=(5000, n_total)).mean(axis=1)
    medias_estratificado = sum(
        pesos_estrato[especie] * generador_diseno.choice(masas_estrato[especie], size=(5000, n_h)).mean(axis=1)
        for especie, n_h in n_estrato.items()
    )
    filas_diseno[n_total] = {
        "EE simple (simulado)": medias_simple.std(),
        "EE simple (fórmula)": sigma / np.sqrt(n_total),
        "EE estratificado (simulado)": medias_estratificado.std(),
        "EE estratificado (fórmula)": np.sqrt(sum(pesos_estrato[e] ** 2 * sigma_estrato[e] ** 2 / n_h for e, n_h in n_estrato.items())),
    }
tabla_diseno = pd.DataFrame(filas_diseno).T
tabla_diseno["efecto de diseño"] = (tabla_diseno["EE estratificado (fórmula)"] / tabla_diseno["EE simple (fórmula)"]) ** 2
tabla_diseno.round(2)

**Interpretación:** con el mismo número de pingüinos, el diseño estratificado tiene un error estándar de **poco más de la mitad** que el aleatorio simple (≈ 59 contra 103 g con *n* = 60), y la simulación coincide con la fórmula. El cociente de varianzas, llamado **efecto de diseño**, es ≈ 0.33: un estratificado con 60 pingüinos es tan preciso como un aleatorio simple con unos 180. En la sesión 1 el EE estratificado salió aún menor (≈ 52 g) porque allí se muestreó **sin reemplazo** de una población de solo 342: medir una fracción grande de la población reduce el error adicionalmente (corrección por población finita).

**Decisión que permite tomar:** si se conoce una variable que explica mucho de lo que se mide (especie, región, nivel socioeconómico), conviene estratificar por ella. **Error común que se evita:** analizar una muestra estratificada con **asignación no proporcional** como si fuera aleatoria simple, sin ponderar por $W_h$; eso sesga la media.

**💡 Ideas clave de la sección 1:**
* Un estadístico varía de muestra en muestra; su distribución es la **distribución muestral** y su desviación estándar, el **error estándar**.
* Ley de los grandes números: $\bar{x}$ se acerca a $\mu$ al crecer $n$. TCL: $\bar{x}$ es aproximadamente normal, más rápido cuanto menos asimétrica sea la población.
* $EE = \sigma / \sqrt{n}$: cuadruplicar $n$ reduce el EE a la mitad.
* El diseño también importa: estratificar por una variable que separa a la población reduce el EE sin aumentar $n$.

### 🟢 Ejercicio 1 · ¿Media o mediana?

**🎯 Objetivo:** comparar la **eficiencia** de dos estimadores del centro, es decir, cuánto varía cada uno de muestra a muestra. **Qué se espera aprender:** que el error estándar no es solo de la media: cada estimador tiene su distribución muestral, y elegir uno es elegir también su precisión.

Con `poblacion`, saca 2,000 muestras de tamaño 25 (usa `generador = np.random.default_rng(401)`) y calcula para cada una la **media** y la **mediana**. ¿Cuál de los dos estimadores varía más de muestra a muestra? Compara sus desviaciones estándar y dibuja ambas distribuciones.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto A · Asignación de Neyman.

**🎯 Objetivo:** explorar cómo repartir una muestra entre estratos. **Qué se espera aprender:** cuándo vale la pena una asignación más sofisticada que la proporcional.

La **asignación de Neyman** reparte la muestra en proporción a $W_h \sigma_h$ en lugar de $W_h$: se muestrean más individuos donde hay más variabilidad. Con *n* = 60, calcula los $n_h$ de Neyman y el EE estratificado con la fórmula, y compáralo con el de la asignación proporcional (sección 1.4). ¿Cuánto se gana?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 2. Error estándar e intervalos de confianza (25 min).

En la práctica solo hay **una** muestra. El error estándar se estima con la desviación estándar muestral: EE = *s* / √*n*. Un **intervalo de confianza** (IC) del 95% para la media es:

```
x̄ ± t(0.975, n − 1) · s / √n
```

Donde:
* `t(0.975, n − 1)` es el cuantil de la distribución *t* de Student con *n* − 1 grados de libertad (≈ 1.98 para *n* = 123).
* `s / √n` es el error estándar estimado.

**Ejemplo:** desde aquí se vuelve a la situación real: los Gentoo medidos son una **muestra** de la población de Gentoo.

**Concepto: de la distribución muestral al intervalo.** Por el TCL, $\bar{x}$ cae a menos de $1.96\,EE$ de $\mu$ en 95% de las muestras. Si eso es cierto, también es cierto que $\mu$ está a menos de $1.96\,EE$ de $\bar{x}$; de ahí sale el intervalo. Como $\sigma$ es desconocida y se estima con $s$, se usa el cuantil de la *t* de Student, un poco mayor que 1.96, para compensar la incertidumbre extra de estimar $\sigma$:

$$IC_{95\%} = \bar{x} \pm t_{0.975,\,n-1}\,\frac{s}{\sqrt{n}}$$

* La siguiente celda calculará el IC del 95% para la masa media de los Gentoo con la fórmula, paso a paso.

In [ ]:
masa_gentoo = pinguinos.loc[pinguinos["species"] == "Gentoo", "body_mass_g"].dropna()
n = len(masa_gentoo)
media = masa_gentoo.mean()
error_estandar = masa_gentoo.std(ddof=1) / np.sqrt(n)
t_critico = stats.t.ppf(0.975, df=n - 1)
n, round(float(media), 1), round(float(error_estandar), 1), np.round([media - t_critico * error_estandar, media + t_critico * error_estandar], 1)

**Interpretación:** con 123 Gentoo, la masa media es 5,076 g con EE de 45.5 g; el IC del 95% va de 4,986 a 5,166 g. El margen (≈ 90 g) es dos errores estándar. Ese margen es mucho menor que la dispersión de los individuos (≈ 500 g), porque describe la incertidumbre sobre **la media**, no sobre cada pingüino.

* La siguiente celda obtendrá el mismo intervalo con `stats.t.interval` de *SciPy* y con `DescrStatsW` de *statsmodels*.

In [ ]:
con_scipy = stats.t.interval(0.95, df=n - 1, loc=media, scale=stats.sem(masa_gentoo))
con_statsmodels = DescrStatsW(masa_gentoo).tconfint_mean(alpha=0.05)
np.round(con_scipy, 1), np.round(con_statsmodels, 1)

**Interpretación:** las dos bibliotecas reproducen exactamente el cálculo a mano. En un informe se usa una de ellas (menos riesgo de errores de escritura), pero conviene saber qué fórmula aplican. `stats.sem` usa `ddof=1` por defecto.

### 2.1 Intervalos para una proporción.

Para una proporción *p̂* = *k* / *n* hay varios métodos (`proportion_confint(k, n, method=...)`):

| Método | `method=` | Comentario |
|---|---|---|
| Wald | `"normal"` | *p̂* ± 1.96·√(*p̂*(1 − *p̂*)/*n*). Simple, pero falla con *n* pequeño o *p̂* cerca de 0 o 1 |
| Wilson | `"wilson"` | Recomendado por defecto: buena cobertura incluso con *n* pequeño |
| Clopper-Pearson | `"beta"` | "Exacto": conservador (intervalos algo más anchos) |

**Concepto:** una proporción es la media de una variable 0/1, así que su error estándar es $EE(\hat{p}) = \sqrt{\hat{p}(1-\hat{p})/n}$. Wald usa directamente $\hat{p} \pm 1.96\,EE$; Wilson "invierte" la prueba *z* y produce un intervalo que nunca sale de [0, 1] y que se centra un poco hacia 0.5; Clopper-Pearson usa la distribución binomial exacta.

* La siguiente celda calculará los tres intervalos para la tasa de supervivencia del Titanic.

In [ ]:
sobrevivientes, pasajeros = int(titanic["survived"].sum()), len(titanic)
pd.DataFrame({metodo: proportion_confint(sobrevivientes, pasajeros, alpha=0.05, method=metodo)
              for metodo in ["normal", "wilson", "beta"]}, index=["inferior", "superior"]).T.round(3)

Con 891 pasajeros los tres métodos coinciden (≈ 35% a 42%). La decisión importa con muestras pequeñas.

**Decisión: ¿Wald o Wilson con pocos datos?** Con 2 éxitos en 30 casos, la fórmula de Wald da un límite inferior **negativo**, imposible para una proporción (`statsmodels` lo recorta a 0, lo que oculta el problema).

* La siguiente celda calculará Wald a mano y Wilson con *statsmodels* para 2 éxitos en 30.

In [ ]:
p_gorro = 2 / 30
margen = 1.96 * np.sqrt(p_gorro * (1 - p_gorro) / 30)
{"Wald a mano": (round(float(p_gorro - margen), 3), round(float(p_gorro + margen), 3)),
 "Wilson": tuple(round(float(v), 3) for v in proportion_confint(2, 30, method="wilson"))}

**Interpretación:** Wald da (−0.023, 0.156), con un límite inferior imposible; Wilson da (0.018, 0.213), asimétrico alrededor de $\hat{p}$ = 0.067 como corresponde a una proporción cercana a 0. **Decisión:** usar Wilson por defecto, sobre todo si $n\hat{p}$ o $n(1-\hat{p})$ es menor que 10.

### 2.2 Consecuencia: ¿cuándo se puede confiar en la fórmula *t*?

El "95%" es una promesa sobre el **procedimiento**: si se repitiera el muestreo, 95% de los intervalos capturarían μ. Esa promesa depende de supuestos. Se simulará la **cobertura real** del IC *t* con dos poblaciones: la masa de los pingüinos (sin colas largas) y la tarifa del Titanic (muy asimétrica, con atípicos).

**Concepto: cobertura.** Es la proporción de intervalos, sobre muchas muestras repetidas, que contienen el parámetro verdadero. Un método "del 95%" que en realidad cubre 80% es engañoso. Solo se puede medir por simulación, cuando se conoce la población.

* La siguiente celda definirá `cobertura_t`, que saca muchas muestras, calcula el IC *t* de cada una y cuenta qué fracción contiene la media poblacional.

In [ ]:
def cobertura_t(poblacion, n, repeticiones=2000, generador=rng):
    muestras = generador.choice(poblacion, size=(repeticiones, n))
    medias = muestras.mean(axis=1)
    errores = muestras.std(axis=1, ddof=1) / np.sqrt(n)
    t_critico = stats.t.ppf(0.975, df=n - 1)
    contiene = (medias - t_critico * errores <= poblacion.mean()) & (poblacion.mean() <= medias + t_critico * errores)
    return contiene.mean()

* La siguiente celda calculará la cobertura con muestras de 10, 30 y 100 en ambas poblaciones (`tarifas` se definió en la sección 1.2).

In [ ]:
pd.DataFrame({n: {"masa (pingüinos)": cobertura_t(poblacion, n), "tarifa (Titanic)": cobertura_t(tarifas, n)}
              for n in [10, 30, 100]}).round(3)

Con la masa, la cobertura es ≈ 95% incluso con *n* = 10. Con la tarifa, un IC "del 95%" con *n* = 10 solo captura la media en torno a 80% de las veces, y aún con *n* = 100 se queda debajo de 95%. **Consecuencia:** con datos muy asimétricos y muestras pequeñas, el intervalo es demasiado estrecho y transmite una seguridad falsa. Conviene reportar la mediana con un IC *bootstrap*, transformar (logaritmo) o, al menos, advertirlo.

**💡 Ideas clave de la sección 2:**
* $IC_{95\%} = \bar{x} \pm t\,s/\sqrt{n}$ para la media; Wilson para proporciones.
* El 95% es una propiedad del **procedimiento**, que solo se cumple si se cumplen sus supuestos.
* Con datos muy asimétricos y $n$ pequeño, el IC *t* cubre menos de lo prometido.

### 🟢 Ejercicio 2 · Supervivencia en tercera clase.

Calcula la tasa de supervivencia de los pasajeros de **tercera clase** con su IC del 95% por Wilson, y compárala con la de primera clase. ¿Se traslapan los intervalos?

**🎯 Objetivo:** calcular IC de proporciones por grupo con `proportion_confint`. **Qué se espera aprender:** a comparar grupos con su incertidumbre, no solo con sus tasas puntuales.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Cobertura de Wald y Wilson.

Simula 5,000 muestras de tamaño 20 de una población con proporción verdadera 0.10 (`generador.binomial(20, 0.10, size=5000)`, con `generador = np.random.default_rng(401)`). Para cada muestra calcula el IC de Wald y el de Wilson y estima la cobertura de cada método. ¿Cuál cumple mejor la promesa del 95%?

**🎯 Objetivo:** medir por simulación la cobertura real de dos métodos. **Qué se espera aprender:** a verificar una afirmación metodológica ("Wilson es mejor") con datos simulados en lugar de aceptarla por autoridad.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 3. *Bootstrap* (25 min).

El *bootstrap* estima la distribución muestral **sin fórmula**: trata la muestra como si fuera la población y la **remuestrea con reemplazo** muchas veces, calculando el estadístico en cada remuestra.

1. Remuestrear *n* valores con reemplazo de la muestra original.
2. Calcular el estadístico (media, mediana, diferencia…).
3. Repetir miles de veces; la dispersión de esos valores aproxima el error estándar y sus percentiles 2.5% y 97.5% dan un IC del 95% (**método percentil**).

* La siguiente celda hará un *bootstrap* a mano de la media de los Gentoo con 9,999 remuestras.

In [ ]:
valores_gentoo = masa_gentoo.to_numpy()
remuestras = rng.choice(valores_gentoo, size=(9999, len(valores_gentoo)), replace=True)
medias_boot = remuestras.mean(axis=1)
round(float(medias_boot.std(ddof=1)), 1), np.percentile(medias_boot, [2.5, 97.5]).round(1)

**Interpretación:** el EE *bootstrap* (≈ 45.3 g) coincide con el de la fórmula (45.5 g), y el IC percentil (≈ 4,987 a 5,165 g) con el IC *t*. Cuando la fórmula es válida, el *bootstrap* la reproduce; su valor está en los casos en que no hay fórmula.

* La siguiente celda dibujará la distribución *bootstrap* con el intervalo percentil.

In [ ]:
ax = sns.histplot(medias_boot, bins=50)
for limite in np.percentile(medias_boot, [2.5, 97.5]):
    ax.axvline(limite, color="black", linestyle="--")
ax.set(xlabel="media bootstrap de la masa de los Gentoo (g)", title="Distribución bootstrap e IC percentil del 95%")
plt.show()

**Interpretación:** la distribución *bootstrap* de la media es simétrica y con forma de campana, como predice el TCL con $n = 123$. Las líneas punteadas marcan los percentiles 2.5 y 97.5, que son los límites del IC.

### 3.1 `scipy.stats.bootstrap`.

```
stats.bootstrap((<muestra>,), <estadistico>, n_resamples=9999, confidence_level=0.95,
                method="BCa", random_state=401)
```

Donde:
* `<muestra>` va dentro de una tupla; para dos grupos se pasa `(<grupo_a>, <grupo_b>)`.
* `<estadistico>` es una función que acepta el argumento `axis` (como `np.mean` o `np.median`).
* `method` puede ser `"percentile"`, `"basic"` o `"BCa"` (por defecto), que corrige sesgo y asimetría.
* `random_state=401` hace reproducible el resultado.

**Concepto: percentil y BCa.** El método percentil toma directamente los percentiles 2.5 y 97.5 de la distribución *bootstrap*. BCa (*bias-corrected and accelerated*) los ajusta para corregir el sesgo del estimador y el cambio de su variabilidad, con mejor cobertura cuando la distribución *bootstrap* es asimétrica.

* La siguiente celda calculará el IC *bootstrap* de la media de los Gentoo con los métodos percentil y BCa.

In [ ]:
{metodo: np.round(stats.bootstrap((valores_gentoo,), np.mean, n_resamples=9999, method=metodo,
                                  random_state=401).confidence_interval, 1)
 for metodo in ["percentile", "BCa"]}

Con 123 Gentoo y una distribución casi simétrica, la fórmula *t* (≈ 4,986 a 5,166 g) y el *bootstrap* difieren en menos de 5 g en cada extremo. La ventaja del *bootstrap* aparece con estadísticos sin fórmula sencilla, como la mediana o una diferencia de medianas.

* La siguiente celda calculará un IC *bootstrap* para la **mediana** de la masa de los Gentoo.

In [ ]:
ic_mediana = stats.bootstrap((valores_gentoo,), np.median, n_resamples=9999, method="percentile", random_state=401)
float(np.median(valores_gentoo)), np.round(ic_mediana.confidence_interval, 1)

**Interpretación:** la mediana de los Gentoo es 5,000 g, con IC de 4,900 a 5,200 g. El intervalo es más ancho que el de la media (≈ 300 contra 180 g), coherente con el ejercicio 1: la mediana es menos eficiente. Sus límites son múltiplos de 50 g porque así se registraron las masas.

### 3.2 Diferencia entre dos grupos.

Para una diferencia, la función recibe dos muestras y el `axis`. Cada grupo se remuestrea **por separado**, respetando su tamaño.

* La siguiente celda definirá `diferencia_medias` y calculará el IC *bootstrap* de la diferencia de masa media entre machos y hembras Gentoo.

In [ ]:
def diferencia_medias(a, b, axis=-1):
    return np.mean(a, axis=axis) - np.mean(b, axis=axis)

gentoo = pinguinos[pinguinos["species"] == "Gentoo"]
machos_gentoo = gentoo.loc[gentoo["sex"] == "Male", "body_mass_g"].to_numpy()
hembras_gentoo = gentoo.loc[gentoo["sex"] == "Female", "body_mass_g"].to_numpy()
ic_diferencia = stats.bootstrap((machos_gentoo, hembras_gentoo), diferencia_medias, n_resamples=9999,
                                method="percentile", random_state=401)
round(float(diferencia_medias(machos_gentoo, hembras_gentoo)), 1), np.round(ic_diferencia.confidence_interval, 1)

**Interpretación:** los machos Gentoo pesan en promedio 805 g más que las hembras, con IC de ≈ 699 a 910 g. El intervalo está lejos de 0: la diferencia no es atribuible a la variación muestral. Este es el formato recomendado para reportar una comparación: **diferencia + IC**, antes que un valor *p* (sesión 6).

**Advertencia:** el *bootstrap* no crea información. Supone que la muestra representa bien a la población y, con muestras pequeñas o asimétricas, también **subcubre**. No corrige un sesgo de selección.

**💡 Ideas clave de la sección 3:**
* El *bootstrap* remuestrea con reemplazo para aproximar la distribución muestral de **cualquier** estadístico.
* `scipy.stats.bootstrap`: percentil o BCa, con `random_state` para reproducir.
* No crea información: no corrige sesgos ni rescata muestras pequeñas y muy asimétricas.

### 🟢 Ejercicio 3 · Mediana de la aleta de los Chinstrap.

Calcula un IC *bootstrap* del 95% (percentil, `random_state=401`) para la **mediana** de `flipper_length_mm` de los Chinstrap. Compáralo con el IC *t* de la media. ¿Por qué el intervalo de la mediana tiene límites "redondos"?

**🎯 Objetivo:** usar `scipy.stats.bootstrap` con la mediana y compararla con la media. **Qué se espera aprender:** a reconocer cómo la forma de registro de los datos (enteros) afecta a la distribución *bootstrap*.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 2 · ¿El *bootstrap* rescata a la tarifa del Titanic?

Repite la simulación de cobertura de la sección 2.2 para la tarifa con *n* = 10, pero con el IC *bootstrap* percentil de la media (usa 300 muestras y 999 remuestras para que no tarde, con `generador = np.random.default_rng(401)`). ¿Mejora la cobertura respecto al IC *t*?

**🎯 Objetivo:** probar los límites del *bootstrap* con una simulación de cobertura. **Qué se espera aprender:** que remuestrear no sustituye a tener una muestra representativa.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## 4. Interpretación correcta de un IC (15 min).

| Afirmación | ¿Correcta? |
|---|---|
| "El procedimiento produce intervalos que capturan la media poblacional en 95% de las muestras repetidas." | ✅ |
| "Con los datos observados, valores de la media entre 4,986 y 5,166 g son compatibles con la muestra." | ✅ (lectura práctica) |
| "Hay 95% de probabilidad de que μ esté entre 4,986 y 5,166 g." | ❌ μ es fija; lo aleatorio es el intervalo |
| "El 95% de los Gentoo pesa entre 4,986 y 5,166 g." | ❌ Confunde el IC de la **media** con la dispersión de los **individuos** |
| "Si dos IC se traslapan, no hay diferencia." | ❌ Pueden traslaparse y la diferencia ser significativa; se calcula el IC **de la diferencia** |
| "Un IC más estrecho siempre es mejor." | ❌ Solo si el procedimiento cumple su cobertura (sección 2.2) |

**¿Por qué esta sección?** Los intervalos de confianza son de los conceptos peor interpretados en la práctica, incluso en artículos publicados. Un informe con buenos cálculos y malas frases transmite conclusiones falsas. La tabla resume las lecturas correctas y los errores más comunes.

**💡 Ideas clave de la sección 4:** un IC describe la incertidumbre sobre un **parámetro**, no la dispersión de los individuos; su 95% se refiere al procedimiento; para comparar grupos se calcula el IC **de la diferencia**.

### 🟢 Ejercicio 4 · Reformula.

Un compañero escribió: *"La masa media de los Gentoo es 5,076 g y hay 95% de probabilidad de que el valor real esté entre 4,986 y 5,166 g, así que casi todos los Gentoo pesan unos 5 kg."* Señala los dos errores y reescribe la frase.

**🎯 Objetivo:** detectar y corregir interpretaciones erróneas de un IC. **Qué se espera aprender:** a redactar resultados con incertidumbre en el estilo que pide el informe del proyecto.

**✍️ Tu respuesta:**

_(escribe aquí)_

### 📣 Proyecto integrador: consigna.

**Se publica hoy en Classroom** (el texto siguiente es la referencia). Forma parte de la **Evaluación semanal 2** (70 de 100 puntos), que cierra el **domingo 1 de noviembre, 11:59 pm (CDMX)**.

**Consigna:** un EDA completo en un *notebook* reproducible en *Codespaces*, más un informe de hallazgos de 2 a 4 páginas (sección Markdown final o HTML/PDF exportado con `nbconvert`).

**Datasets a elegir:** `diamonds` (*Seaborn*), Adult / Census Income (UCI), Bike Sharing (UCI), Wine Quality (UCI), o uno propio **aprobado por el instructor antes del martes 27 de octubre**.

**Requisitos mínimos:**
1. 2 o 3 preguntas de análisis.
2. Auditoría de calidad y tratamiento justificado.
3. Descripción de distribuciones y atípicos.
4. Al menos una comparación de grupos y una relación entre variables.
5. Al menos dos pruebas de hipótesis distintas (o una prueba y un *bootstrap*), con supuestos revisados y tamaño del efecto.
6. Informe con resumen ejecutivo y limitaciones.

**Entrega en Classroom:** el `.ipynb` ejecutado, el informe (`.html` o `.pdf`) y el enlace al repositorio de GitHub. En la sesión 7 se entrega la **plantilla** `proyecto/plantilla_proyecto_integrador.ipynb` con esta estructura ya armada.

**Rúbrica del Proyecto integrador (70 puntos):**

| Criterio | Puntos | Excelente |
|---|---|---|
| Preguntas y contexto | 6 | Preguntas claras y respondibles con los datos; fuente y sesgos posibles descritos |
| Calidad de datos | 10 | Auditoría completa; decisiones de limpieza e imputación justificadas y con su impacto evaluado |
| Descripción y visualización | 12 | Distribuciones, atípicos y relaciones bien descritos; gráficas efectivas y rotuladas |
| Inferencia | 16 | Pruebas adecuadas, supuestos revisados, IC o *bootstrap*, tamaño del efecto interpretado |
| Informe de hallazgos | 16 | Resumen ejecutivo claro; hallazgos en lenguaje llano con su incertidumbre; limitaciones; sin causalidad injustificada |
| Reproducibilidad | 10 | Corre de principio a fin en *Codespaces*; `requirements.txt`, semillas y estructura ordenada |

Escala por criterio: Excelente = 100% de los puntos, Suficiente ≈ 60%, Insuficiente = 0 a 30%.

---
## 5. Ejercicio de la sesión (20 min).

### 🟢 Ejercicio 5 · Masa media de los Gentoo: fórmula *t* y *bootstrap*.

**🎯 Objetivo:** integrar en una tabla los tres métodos de la sesión. **Qué se espera aprender:** a decidir cuándo los métodos coinciden y cuál reportar.

Reporta en una tabla la masa media de los Gentoo con su IC del 95% por la fórmula *t* y por *bootstrap* (percentil y BCa, `random_state=401`), junto con el ancho de cada intervalo.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

### 🟢 Ejercicio 6 · Diferencia de medianas entre machos y hembras Gentoo.

Calcula la diferencia de **medianas** de masa entre machos y hembras Gentoo con su IC *bootstrap* percentil del 95%. ¿El intervalo incluye el 0? ¿Qué significa?

**🎯 Objetivo:** construir un IC para un estadístico sin fórmula sencilla. **Qué se espera aprender:** a leer si un IC de una diferencia incluye el 0 y qué implica.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 3 · ¿Y si se mezclan las especies?

Repite el ejercicio 6 con **todos** los pingüinos (sin separar por especie) y después dentro de cada especie. ¿Por qué la diferencia global no coincide con ninguna de las diferencias por especie? ¿Cuál reportarías?

**🎯 Objetivo:** ver cómo una variable de agrupación cambia una comparación. **Qué se espera aprender:** a decidir el nivel de agregación adecuado antes de reportar una diferencia.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 7 · Comenta los resultados.

En 3 o 4 frases: ¿coinciden la fórmula *t* y el *bootstrap* para la media de los Gentoo? ¿Por qué? ¿Qué aporta el *bootstrap* en el ejercicio 6 que no daba la fórmula?

**🎯 Objetivo:** redactar conclusiones con incertidumbre. **Qué se espera aprender:** a justificar la elección de método con los supuestos.

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Hoy vimos:**
* Distribución muestral, ley de los grandes números y teorema central del límite (lento con datos muy asimétricos).
* Error estándar: EE = σ / √*n*; cuadruplicar *n* reduce el EE a la mitad; estratificar reduce el EE sin aumentar *n*.
* IC *t* para la media (*SciPy* y *statsmodels*) e IC para proporciones (Wald, Wilson, Clopper-Pearson).
* La cobertura real depende de los supuestos: con datos muy asimétricos y *n* pequeño, el IC "del 95%" cubre bastante menos.
* *Bootstrap* con `scipy.stats.bootstrap`: percentil y BCa, para la media, la mediana y diferencias entre grupos.
* Cómo interpretar (y cómo no) un intervalo de confianza.

**Práctica (sin calificación):** terminar los ejercicios 🟢 y elegir el *dataset* del Proyecto integrador. Si es uno propio, pedir aprobación antes del martes 27 de octubre.

**Próxima sesión (mar 27 oct):** pruebas de hipótesis y tamaño del efecto (`06_pruebas_hipotesis_efecto.ipynb`).

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>